# Notebook 03 — Privacy, Governance & Compliance Audit
## NovaCred Credit Application Dataset
### Role: Data Governance Officer

---

This notebook is the formal governance and compliance audit for NovaCred's credit application pipeline. It is structured around four responsibilities of the Governance Officer role:

1. **PII Identification & Classification** — cataloguing every personally identifiable field and its risk level
2. **Pseudonymization Demonstration** — implementing and critically evaluating privacy-preserving techniques in code
3. **GDPR Compliance Mapping** — mapping observed pipeline gaps to specific regulatory obligations
4. **EU AI Act Classification & Bias Governance** — classifying the system under the EU AI Act
5. **Governance Controls & Recommendations** - proposing actionable controls
6. **Executive Summary** - summarizing key insights and requiredf actions


This audit builds on the findings of `01-data-quality.ipynb` (data engineering) and `02-data-analysis.ipynb` (bias detection) and interpreting them through a regulatory and governance lens.

> **Regulatory Scope:** GDPR (EU) 2016/679 · EU AI Act (EU) 2024/1689

## Setup

In [10]:
import pandas as pd
import numpy as np
import hashlib
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the cleaned dataset produced by 01-data-quality.ipynb
df = pd.read_csv('../data/clean_credit_applications.csv')
print(f"Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns")

Dataset loaded: 485 records, 33 columns


---
## Part 1 — PII Identification & Classification

### Regulatory Foundation

The GDPR (Article 4(1)) defines personal data as any information relating to an identified or identifiable natural person. This definition is intentionally broad and creates two distinct risk tiers that determine how data must be treated throughout its lifecycle.

**Direct Identifiers** are fields that identify an individual on their own, without any additional information. Their exposure immediately and unambiguously ties back to a real person.

**Quasi-Identifiers** are fields that appear innocuous in isolation but become identifying when combined. A landmark study by Sweeney (2000) demonstrated that ZIP code, gender, and date of birth in combination are sufficient to uniquely identify approximately 87% of the US population — all three of these fields are present in this dataset.

The distinction matters operationally: direct identifiers must be pseudonymised or dropped for any analytical use; quasi-identifiers require generalisation techniques such as age banding or ZIP truncation to reduce re-identification risk.

In [2]:
import pandas as pd

pii_catalog = pd.DataFrame([
    # ------------------ DIRECT IDENTIFIERS ------------------
    {
     "Field": "applicant_info.full_name",  
     "Type": "Direct Identifier",
     "Sensitivity": "HIGH",
     "Present in Clean CSV": "Yes — 489 plaintext values",
     "GDPR Concern": "Art. 4(1): identifies individual on its own",
     "Recommended Action": "Tokenize via secured lookup table. WHY: Names must be masked for analysis, but the business must retain the ability to reverse the token if the user submits a GDPR Art. 15 (Access) or Art. 17 (Erasure) request."
    },
    {
     "Field": "applicant_info.ssn",        
     "Type": "Direct Identifier",
     "Sensitivity": "CRITICAL",
     "Present in Clean CSV": "Yes — 489 plaintext values",
     "GDPR Concern": "Art. 4(1): government-issued unique ID",
     "Recommended Action": "Apply Salted SHA-256 hashing. WHY: SSNs have a predictable format, making plain hashes vulnerable to pre-computation attacks. Salting secures the hash, and since SSNs aren't needed for operational recovery, a one-way hash is ideal."
    },
    {
     "Field": "applicant_info.email",      
     "Type": "Direct Identifier",
     "Sensitivity": "HIGH",
     "Present in Clean CSV": "Yes — 487 plaintext values",
     "GDPR Concern": "Art. 4(1): directly identifies individual",
     "Recommended Action": "Drop or apply Salted SHA-256 hashing. WHY: Email has no predictive value for credit scoring. Hashing allows data engineers to check for duplicate applications without exposing the actual email address."
    },
    {
     "Field": "applicant_info.ip_address", 
     "Type": "Direct Identifier",
     "Sensitivity": "HIGH",
     "Present in Clean CSV": "Yes — 489 plaintext values",
     "GDPR Concern": "Art. 4(1): ECJ (Breyer) confirms IP = personal data",
     "Recommended Action": "Drop from analytical datasets. WHY: IPs are only relevant for operational security/fraud logs. They have no place in a dataset used to train a credit model (violates Data Minimisation)."
    },

    # ------------------ QUASI-IDENTIFIERS ------------------
    {
     "Field": "applicant_info.date_of_birth", 
     "Type": "Quasi-Identifier",
     "Sensitivity": "MEDIUM",
     "Present in Clean CSV": "Yes — 489 exact birth dates",
     "GDPR Concern": "Art. 4(1): part of re-identifying combination",
     "Recommended Action": "Generalise to age bands (e.g., 18-25, 26-40). WHY: Exact DOB combined with ZIP and Gender creates an 87% risk of re-identification. Banding preserves the analytical value (age risk) while destroying the exact identifying combination."
    },
    {
     "Field": "applicant_info.zip_code",   
     "Type": "Quasi-Identifier + Proxy",
     "Sensitivity": "MEDIUM",
     "Present in Clean CSV": "Yes — 489 five-digit values",
     "GDPR Concern": "Art. 4(1): re-identifying combination",
     "Recommended Action": "Truncate to 3-digit prefix. WHY: Full ZIP codes are highly granular. Truncating reduces re-identification risk while keeping regional data. It also mitigates the risk of illegal 'redlining' (proxy discrimination based on neighborhood)."
    },
    {
     "Field": "applicant_info.gender",     
     "Type": "Quasi-Identifier + Protected",
     "Sensitivity": "MEDIUM",
     "Present in Clean CSV": "Yes",
     "GDPR Concern": "Art. 4(1) and EU AI Act (bias risk)",
     "Recommended Action": "Isolate and restrict access. WHY: Using gender for credit decisions violates anti-discrimination laws. However, it MUST be kept in a separate, secured table to perform mandatory bias audits under the EU AI Act (Art. 10)."
    },

    # ------------------ BEHAVIOURAL DATA ------------------
    {
     "Field": "spending_Adult Entertainment", 
     "Type": "Sensitive Behavioural",
     "Sensitivity": "HIGH",
     "Present in Clean CSV": "Yes — 5 records",
     "GDPR Concern": "Art. 5(1)(b): Purpose Limitation",
     "Recommended Action": "Drop entirely. WHY: Under GDPR Art. 5(1)(c) Data Minimisation, you cannot process sensitive lifestyle data if it has no scientifically proven, documented relevance to creditworthiness."
    },
    {
     "Field": "spending_Gambling",         
     "Type": "Sensitive Behavioural",
     "Sensitivity": "HIGH",
     "Present in Clean CSV": "Yes — 6 records",
     "GDPR Concern": "Art. 5(1)(b): Purpose Limitation",
     "Recommended Action": "Drop entirely. WHY: Unless NovaCred has a documented, legally defensible model proving gambling strictly correlates with loan default, collecting this violates purpose limitation."
    },
    {
     "Field": "spending_Alcohol",          
     "Type": "Sensitive Behavioural",
     "Sensitivity": "MEDIUM-HIGH",
     "Present in Clean CSV": "Yes — 11 records",
     "GDPR Concern": "Art. 5(1)(b): Purpose Limitation",
     "Recommended Action": "Drop entirely. WHY: Highly subjective and risks proxy discrimination. Purchasing alcohol does not mathematically preclude someone from paying a mortgage."
    }
])

pd.set_option("display.max_colwidth", 150) # Increased so the explanations aren't cut off!
pii_catalog[["Field","Type", "Sensitivity", "Present in Clean CSV","GDPR Concern", "Recommended Action"]]

,Field,Type,Sensitivity,Present in Clean CSV,GDPR Concern,Recommended Action
0,applicant_info.full_name,Direct Identifier,HIGH,Yes — 489 plaintext values,Art. 4(1): identifies individual on its own,"Tokenize via secured lookup table. WHY: Names must be masked for analysis, but the business must retain the ability to reverse the token if the us..."
1,applicant_info.ssn,Direct Identifier,CRITICAL,Yes — 489 plaintext values,Art. 4(1): government-issued unique ID,"Apply Salted SHA-256 hashing. WHY: SSNs have a predictable format, making plain hashes vulnerable to pre-computation attacks. Salting secures the ..."
2,applicant_info.email,Direct Identifier,HIGH,Yes — 487 plaintext values,Art. 4(1): directly identifies individual,Drop or apply Salted SHA-256 hashing. WHY: Email has no predictive value for credit scoring. Hashing allows data engineers to check for duplicate ...
3,applicant_info.ip_address,Direct Identifier,HIGH,Yes — 489 plaintext values,Art. 4(1): ECJ (Breyer) confirms IP = personal data,Drop from analytical datasets. WHY: IPs are only relevant for operational security/fraud logs. They have no place in a dataset used to train a cre...
4,applicant_info.date_of_birth,Quasi-Identifier,MEDIUM,Yes — 489 exact birth dates,Art. 4(1): part of re-identifying combination,"Generalise to age bands (e.g., 18-25, 26-40). WHY: Exact DOB combined with ZIP and Gender creates an 87% risk of re-identification. Banding preser..."
5,applicant_info.zip_code,Quasi-Identifier + Proxy,MEDIUM,Yes — 489 five-digit values,Art. 4(1): re-identifying combination,Truncate to 3-digit prefix. WHY: Full ZIP codes are highly granular. Truncating reduces re-identification risk while keeping regional data. It als...
6,applicant_info.gender,Quasi-Identifier + Protected,MEDIUM,Yes,Art. 4(1) and EU AI Act (bias risk),"Isolate and restrict access. WHY: Using gender for credit decisions violates anti-discrimination laws. However, it MUST be kept in a separate, sec..."
7,spending_Adult Entertainment,Sensitive Behavioural,HIGH,Yes — 5 records,Art. 5(1)(b): Purpose Limitation,"Drop entirely. WHY: Under GDPR Art. 5(1)(c) Data Minimisation, you cannot process sensitive lifestyle data if it has no scientifically proven, doc..."
8,spending_Gambling,Sensitive Behavioural,HIGH,Yes — 6 records,Art. 5(1)(b): Purpose Limitation,"Drop entirely. WHY: Unless NovaCred has a documented, legally defensible model proving gambling strictly correlates with loan default, collecting ..."
9,spending_Alcohol,Sensitive Behavioural,MEDIUM-HIGH,Yes — 11 records,Art. 5(1)(b): Purpose Limitation,Drop entirely. WHY: Highly subjective and risks proxy discrimination. Purchasing alcohol does not mathematically preclude someone from paying a mo...


### PII Sensitivity Classification Framework

The severity categories assigned in the PII Catalog above are derived from a risk-based approach aligned with the GDPR and the EU AI Act. The classification evaluates two factors: the potential harm to the individual in the event of unauthorized disclosure (e.g., identity theft, discrimination, reputational damage) and the regulatory risk to NovaCred.

* **CRITICAL: Irrevocable Government Identifiers**
    * **Criteria:** Data that uniquely identifies an individual across multiple external systems and cannot be easily changed if compromised.
    * **Examples in Dataset:** Social Security Number (SSN).
    * **Justification:** A breach of this data presents an immediate, catastrophic risk of financial fraud and identity theft. It carries the highest regulatory penalty and requires the strictest cryptographic protection (e.g., one-way salted hashing).

* **HIGH: Direct Identifiers & Highly Stigmatized Data**
    * **Criteria:** Data that directly identifies a specific natural person, or behavioral data that carries severe reputational or discriminatory risk if exposed. 
    * **Examples in Dataset:** Full Name, Email, IP Address, Adult Entertainment/Gambling transactions.
    * **Justification:** Direct identifiers easily expose an individual's identity without requiring cross-referencing. Highly stigmatized transaction data poses a severe risk to the subject's personal life and violates the GDPR principle of data minimization if collected without a strict, documented mathematical need for credit scoring.

* **MEDIUM-HIGH: Subjective Lifestyle Data**
    * **Criteria:** Data that does not directly identify a person, but reveals lifestyle choices that could be misused for proxy discrimination.
    * **Examples in Dataset:** Alcohol expenditures.
    * **Justification:** While not as stigmatized as gambling, this data introduces high regulatory risk. It has dubious predictive value for creditworthiness and opens the system up to subjective bias, requiring strict justification for retention.

* **MEDIUM: Quasi-Identifiers & Protected Attributes**
    * **Criteria:** Data fields that seem innocuous in isolation but become highly identifying when combined (the "Mosaic Effect"), or attributes protected by anti-discrimination laws.
    * **Examples in Dataset:** Date of Birth, ZIP Code, Gender.
    * **Justification:** As demonstrated by the Sweeney study, the combination of ZIP, Gender, and DOB can uniquely identify 87% of the population. While these fields contain highly valuable analytical signals (unlike Names or SSNs), they must be treated with privacy-enhancing techniques like generalization (age banding) or truncation to safely balance utility and privacy. Furthermore, Gender requires special handling to comply with the EU AI Act's mandatory bias auditing.

---
### 1.1 — Quasi-Identifier Re-identification Risk

Removing the four direct identifiers is a necessary first step — but it is not sufficient. The cell below tests empirically whether the quasi-identifiers alone can re-identify applicants, using the methodology used in Sweeney (2000).

In [12]:
quasi_cols  = ["applicant_info.zip_code", "applicant_info.gender", "applicant_info.date_of_birth"]
df_quasi    = df[quasi_cols].dropna()
total       = len(df_quasi)
unique      = df_quasi.drop_duplicates().shape[0]
re_id_rate  = unique / total * 100

print(f"Records with all three quasi-identifiers : {total}")
print(f"Unique (ZIP, Gender, DOB) combinations   : {unique}")
print(f"Re-identification rate                   : {re_id_rate:.1f}%")

Records with all three quasi-identifiers : 485
Unique (ZIP, Gender, DOB) combinations   : 485
Re-identification rate                   : 100.0%


> **Finding:** Every applicant in this dataset is uniquely identifiable using only ZIP code, gender, and date of birth — a 100% re-identification rate. Therefore, Quasi-identifier **generalisation** (age banding, ZIP truncation) is also required before any analytical export. Both are applied in Section 2.

---
## Part 2 — Pseudonymization Demonstration

### Regulatory Foundation

GDPR Article 4(5) defines pseudonymisation as processing personal data so it can no longer be attributed to a specific data subject without additional information kept separately. Three techniques exist — the choice depends on whether the original value needs to be recoverable for authorised purposes (e.g., fulfilling a GDPR Art. 17 erasure request):

| Technique | Reversible? | Still Personal Data? | Key Weakness |
|-----------|-------------|---------------------|--------------|
| **Plain SHA-256 hashing** | No | Yes | **Deterministic** — same input always yields same output. An attacker who knows the input space (e.g. all valid SSN formats) can pre-compute every possible hash and match against the dataset. No lookup table required for the attack. |
| **Salted SHA-256 hashing** | No (without salt) | Yes | Salt must be stored securely and separately. If the salt is exposed alongside the data, the protection collapses. Appropriate when recovery is not operationally needed. |
| **Tokenization** | Yes (via lookup table) | Yes | Lookup table must be secured; if stolen, all tokens are reversed. Enables GDPR Art. 17 erasure: delete the token from the table and the record becomes effectively anonymous. |

**Decision applied in this notebook:**
- **SSN, email, IP address** → Salted SHA-256. These fields have no business reason to be recovered after pseudonymisation. The salt makes pre-computation attacks infeasible.
- **Full name** → Tokenization. May need to be recovered by a compliance officer responding to a GDPR Art. 15 access or Art. 17 erasure request. A secured lookup table enables this without exposing the name in the analytical dataset.

---
## Part 3 — GDPR Compliance Mapping

### Regulatory Foundation

The GDPR establishes seven data protection principles in Article 5(1) that govern every stage of data processing. Compliance must be demonstrable at every stage of the data lifecycle — from collection through to deletion. Article 5(2) places the burden of proof on the data controller: NovaCred must be able to prove compliance, not merely claim it.

The audit below maps each relevant GDPR principle and article to the specific gaps identified in the NovaCred dataset and pipeline.

---
## Part 4 — EU AI Act Classification & Governance Controls

### Regulatory Foundation

The EU AI Act (Regulation 2024/1689), which entered into force in August 2024, establishes a risk-based classification framework for AI systems. **Credit scoring and creditworthiness assessment** is explicitly enumerated as a **High-Risk AI system** in Annex III, Point 5(b). This classification applies directly to NovaCred's automated loan approval system.

High-risk AI systems are subject to mandatory obligations under Chapter III, Section 2, including:

| Article | Obligation |
|---------|-----------|
| Art. 9  | A risk management system must be established and maintained throughout the lifecycle |
| Art. 10 | Training, validation, and testing data must meet quality criteria including examination for biases |
| Art. 13 | The system must be sufficiently transparent to enable the deployer to interpret its output |
| Art. 14 | The system must allow effective human oversight, including the ability to override outputs |
| Art. 15 | The system must maintain adequate accuracy, robustness, and cybersecurity with documented metrics |
| Art. 43 | The system must undergo a conformity assessment before deployment |

---
## Governance Controls & Recommendations



---
## Executive Summary
